# Week 4 — Heuristic Scoring Notebook

This notebook converts wallet-level features into:
- `human_score` (0–100)
- `human_likelihood` (`High` / `Medium` / `Low`)
- `trust_tier` (`Gold` / `Silver` / `Bronze`)

## Input
- `data/feature_table.csv`

## Output
- `data/scoring_output.csv`

The scoring system is heuristic and interpretable, which makes it suitable for an MVP.


## 0) Imports + Load Data

In [ ]:
import pandas as pd
import numpy as np

features = pd.read_csv('data/feature_table.csv')
features.head()

## 1) Check Feature Columns

In [ ]:
features.columns.tolist()

## 2) Scoring Rules

Each wallet starts with a baseline score of **50**.

Then the score is adjusted using interpretable rules:
- older wallets score higher
- bursty or extreme behavior scores lower
- diverse interaction patterns score higher


In [ ]:
def score_wallet(row):
    score = 50
    reasons = []

    # Wallet age
    if row['wallet_age_days'] >= 365:
        score += 15
        reasons.append('older wallet (+15)')
    elif row['wallet_age_days'] >= 90:
        score += 8
        reasons.append('moderately aged wallet (+8)')
    elif row['wallet_age_days'] < 30:
        score -= 12
        reasons.append('very new wallet (-12)')

    # Activity spread
    if row['active_days'] >= 30:
        score += 10
        reasons.append('activity spread across many days (+10)')
    elif row['active_days'] < 3:
        score -= 10
        reasons.append('very few active days (-10)')

    # Transaction frequency
    if row['tx_frequency'] > 5:
        score -= 15
        reasons.append('extremely high tx frequency (-15)')
    elif row['tx_frequency'] > 1:
        score -= 5
        reasons.append('high tx frequency (-5)')
    elif 0.01 <= row['tx_frequency'] <= 1:
        score += 5
        reasons.append('reasonable tx frequency (+5)')

    # Burst behavior
    if row['burst_tx_ratio'] > 0.50:
        score -= 20
        reasons.append('very bursty behavior (-20)')
    elif row['burst_tx_ratio'] > 0.20:
        score -= 10
        reasons.append('moderately bursty behavior (-10)')
    else:
        score += 5
        reasons.append('low burst behavior (+5)')

    # Counterparty diversity
    if row['unique_counterparties'] >= 10:
        score += 10
        reasons.append('high counterparty diversity (+10)')
    elif row['unique_counterparties'] <= 2:
        score -= 10
        reasons.append('very low counterparty diversity (-10)')

    # Interaction entropy
    if row['interaction_entropy'] >= 2:
        score += 8
        reasons.append('diverse interaction distribution (+8)')
    elif row['interaction_entropy'] < 0.5:
        score -= 8
        reasons.append('concentrated interaction pattern (-8)')

    # Time-gap pattern
    if row['mean_time_gap_hours'] > 0 and row['mean_time_gap_hours'] < 0.1:
        score -= 10
        reasons.append('extremely short average time gaps (-10)')
    elif row['mean_time_gap_hours'] >= 1:
        score += 5
        reasons.append('reasonable average time gaps (+5)')

    if row['std_time_gap_hours'] == 0:
        score -= 5
        reasons.append('no timing variation (-5)')
    elif row['std_time_gap_hours'] > 1:
        score += 3
        reasons.append('natural timing variation (+3)')

    # Token diversity
    if 'unique_token_symbols' in row.index:
        if row['unique_token_symbols'] >= 5:
            score += 5
            reasons.append('multiple token interactions (+5)')
        elif row['unique_token_symbols'] == 0:
            score -= 2
            reasons.append('no token diversity (-2)')

    score = max(0, min(100, score))
    return pd.Series({'human_score': score, 'score_reasons': '; '.join(reasons)})

## 3) Apply Scoring Function

In [ ]:
scored = features.copy()
scored[['human_score', 'score_reasons']] = scored.apply(score_wallet, axis=1)
scored[['wallet', 'human_score', 'score_reasons']].head()

## 4) Convert Score into Human Likelihood + Trust Tier

In [ ]:
def human_likelihood(score):
    if score >= 75:
        return 'High'
    elif score >= 50:
        return 'Medium'
    return 'Low'

def trust_tier(score):
    if score >= 80:
        return 'Gold'
    elif score >= 60:
        return 'Silver'
    return 'Bronze'

scored['human_likelihood'] = scored['human_score'].apply(human_likelihood)
scored['trust_tier'] = scored['human_score'].apply(trust_tier)

scored[['wallet', 'human_score', 'human_likelihood', 'trust_tier']].head()

## 5) Review Output Distribution

In [ ]:
scored['human_score'].describe()

In [ ]:
scored['human_likelihood'].value_counts(dropna=False)

In [ ]:
scored['trust_tier'].value_counts(dropna=False)

## 6) Inspect Highest- and Lowest-Scoring Wallets

In [ ]:
scored.sort_values('human_score', ascending=False)[['wallet', 'human_score', 'human_likelihood', 'trust_tier', 'score_reasons']].head(5)

In [ ]:
scored.sort_values('human_score', ascending=True)[['wallet', 'human_score', 'human_likelihood', 'trust_tier', 'score_reasons']].head(5)

## 7) Save Final Output

In [ ]:
output_cols = ['wallet', 'human_score', 'human_likelihood', 'trust_tier', 'score_reasons']
scored[output_cols].to_csv('data/scoring_output.csv', index=False)
print('✅ Saved: data/scoring_output.csv')
scored[output_cols].head()

## 8) Next Steps

After this notebook, you can:
- refine weights and thresholds
- add simulated bot wallets for validation
- expose results through an API endpoint
- build a dashboard to inspect score outputs
